In [ ]:
import pickle
from pathlib import Path
from copy import deepcopy
from functools import partial

import numpy as np
import pandas as pd
import torch
import multiprocessing as mp
from lightning.pytorch import (
    seed_everything,
)
import matplotlib.pyplot as plt
from msas_pytorch import msas
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import seaborn as sns
from patientflow.data import BrainteaserDataModule
from patientflow.metrics import PrognosisMetric

In [ ]:
SEED = 42
seed_everything(SEED)

In [ ]:
real = pd.read_csv(
    Path.home()
    / "Research"
    / "Virtual-ALS-Patients"
    / "data"
    / "no_labels_no_birth_year"
    / "train.csv"
)
real.head()

In [ ]:
# filter out patients with no temporal data (all unknowns)

dfs = []

for _, pdf in real.groupby(by="REF"):
    if (pdf[[f"P{j}" for j in range(1, 13)]] == -1.0).all().all():
        continue
    dfs.append(pdf)

real = pd.concat(dfs, ignore_index=True)

In [ ]:
real_ds = BrainteaserDataModule(
    SEED,
    32,
    Path.home()
    / "Research"
    / "Virtual-ALS-Patients"
    / "data"
    / "no_labels_no_birth_year",
)
real_ds.setup("fit")
real_test_ds = real_ds.test_dataset
real_train_ds = real_ds.train_dataset

In [ ]:
discrete_temporal_features_indices = torch.LongTensor(
    real_ds.features.temporal_features().categorical_features_indices()
)
discrete_temporal_features_num_categories = torch.LongTensor(
    real_ds.features.temporal_features().categorical_features_num_categories()
)
discrete_static_features_indices = torch.LongTensor(
    real_ds.features.static_features().categorical_features_indices()
)
discrete_static_features_num_categories = torch.LongTensor(
    real_ds.features.static_features().categorical_features_num_categories()
)

In [ ]:
version = "2025-03-28 15:06:33"
fname = f"synthetic_dfs_n_samples=100_patientflow_{version}_seed=42.pkl"

with open(fname, "rb") as f:
    gen_dfs, categorical_feats_info = pickle.load(f)

In [ ]:
len(real_ds.train_dataset)

In [ ]:
def calculate_msas(gen_df: pd.DataFrame, ds) -> float:
    (synthetic_static_tensor, synthetic_temporal_tensor, synthetic_seq_lengths) = (
        ds.patient_dfs_to_tensors(
            ds.encode_features(
                gen_df,
                reassign_seq_lengths_weights=True,
                requires_median_delta_calc=False,
            )
        )
    )
    try:
        synthetic_temporal_tensor[:, 0, -1] = 0.0
        return msas(
            real_train_ds[:][1].cpu(),
            synthetic_temporal_tensor.cpu(),
            [torch.mean, torch.std, torch.median, torch.max, torch.min],
            discrete_temporal_features_indices=discrete_temporal_features_indices
            if discrete_temporal_features_indices.size(0)
            else None,
            discrete_temporal_features_num_categories=discrete_temporal_features_num_categories
            if discrete_temporal_features_num_categories.size(0)
            else None,
            real_static_data=real_train_ds[:][0].cpu(),
            synthetic_static_data=synthetic_static_tensor.cpu(),
            discrete_static_features_indices=discrete_static_features_indices
            if discrete_static_features_indices.size(0)
            else None,
            discrete_static_features_num_categories=discrete_static_features_num_categories
            if discrete_static_features_num_categories.size(0)
            else None,
            enforce_temporal_shape=False,
        ).item()

    except Exception as e:
        print(real_train_ds[:][0].size())
        print(synthetic_static_tensor.size())
        raise

In [ ]:
parallel = Parallel(n_jobs=-1, verbose=10)
msas_scores = np.array(
    parallel(delayed(calculate_msas)(gen_df, deepcopy(real_ds)) for gen_df in gen_dfs)
)

In [ ]:
print(f"Min: {msas_scores.min()}")
print(f"25th percentile: {np.percentile(msas_scores, 25)}")
print(f"Median: {np.median(msas_scores)}")
print(f"75th percentile: {np.percentile(msas_scores, 75)}")
print(f"Max: {msas_scores.max()}")
print(f"Mean: {msas_scores.mean()}")
print(f"Std: {msas_scores.std()}")
print()

In [ ]:
def process_endpoint_real_data(endpoint, real_ds, num_runs=10):
    """
    Process all runs for a single endpoint.
    
    Parameters:
    -----------
    endpoint : int
        The endpoint number to process
    real_ds : object
        Dataset object containing train and test datasets
    num_runs : int, default=10
        Number of metric runs to perform for this endpoint
        
    Returns:
    --------
    list
        List of metrics for this endpoint
    """
    seed_everything(SEED)
    metrics = []

    with tqdm(total=num_runs, desc=f"Endpoint {endpoint}", position=endpoint-1, leave=True) as pbar:
        for i in range(num_runs):
            
            test_ds = real_ds.df_to_endpoint_tensor_dataset(
                real_ds.sample_to_df(
                    real_ds.test_dataset[:][0].cpu(),
                    real_ds.test_dataset[:][1].cpu(),
                    real_ds.test_dataset[:][-1].cpu(),
                    transform_categorical=False,
                    transform_continuous=False,
                ),
                180,
                endpoint,
                scale_time_delta=True,
            )
            
            train_ds = real_ds.df_to_endpoint_tensor_dataset(
                real_ds.sample_to_df(
                    real_ds.train_dataset[:][0].cpu(),
                    real_ds.train_dataset[:][1].cpu(),
                    real_ds.train_dataset[:][-1].cpu(),
                    transform_categorical=False,
                    transform_continuous=False,
                ),
                180,
                endpoint,
                scale_time_delta=True,
            )

            p_metric = PrognosisMetric.compute(
                train_ds,
                test_ds,
                enable_progress_bar=False,
                data_loader_num_workers=mp.cpu_count() // 5,
                data_loader_context="fork",
            )
            metrics.append(p_metric)
            
            # Only print dataset stats on the first run for this endpoint
            if i == 0:
                print(f"\n[C{endpoint}] Train Positive Percentage: {train_ds[:][3].sum() / len(train_ds)}")
                print(f"[C{endpoint}] Train Dataset Size: {len(train_ds)}")
                print(f"[C{endpoint}] Train Mean Timesteps: {train_ds[:][2].float().mean().item()}")
                print(f"[C{endpoint}] Test Positive Percentage: {test_ds[:][3].sum() / len(test_ds)}")
                print(f"[C{endpoint}] Test Dataset Size: {len(test_ds)}")
                print(f"[C{endpoint}] Test Mean Timesteps: {test_ds[:][2].float().mean().item()}")
                print()
            
            # Update progress bar
            pbar.update(1)
            
    return metrics, endpoint

In [ ]:
NUM_REAL_METRIC_RUNS = 10

parallel = Parallel(n_jobs=5, verbose=1)
process_func = partial(process_endpoint_real_data, num_runs=NUM_REAL_METRIC_RUNS)
results = parallel(
    delayed(process_func)(endpoint, real_ds) for endpoint in range(1, 6)
)

real_prognostic_metrics = {endpoint: metrics for metrics, endpoint in results}

In [ ]:
with open("real_prognostic_metrics.pkl", "wb") as f:
    pickle.dump(real_prognostic_metrics, f)

In [ ]:
with open("real_prognostic_metrics.pkl", "rb") as f:
    real_prognostic_metrics = pickle.load(f)

In [ ]:
for endpoint in range(1, 6):
    print(f"[C{endpoint}] Metric Scores: {real_prognostic_metrics[endpoint]}")
    print(f"[C{endpoint}] Min: {np.min(real_prognostic_metrics[endpoint])}")
    print(
        f"[C{endpoint}] 25th percentile: {np.percentile(real_prognostic_metrics[endpoint], 25)}"
    )
    print(f"[C{endpoint}] Median: {np.median(real_prognostic_metrics[endpoint])}")
    print(
        f"[C{endpoint}] 75th percentile: {np.percentile(real_prognostic_metrics[endpoint], 75)}"
    )
    print(f"[C{endpoint}] Max: {np.max(real_prognostic_metrics[endpoint])}")
    print(f"[C{endpoint}] Mean: {np.mean(real_prognostic_metrics[endpoint])}")
    print(f"[C{endpoint}] Std: {np.std(real_prognostic_metrics[endpoint])}")
    print()

In [ ]:
def process_endpoint_synthetic_data(endpoint, gen_dfs, real_ds):
    """
    Process all dataframes for a single endpoint.
    
    Parameters:
    -----------
    endpoint : int
        The endpoint number to process
    gen_dfs : list
        List of dataframes to process
    real_ds : object
        Dataset object containing methods and test dataset
        
    Returns:
    --------
    tuple
        (metrics_array, success_array) for this endpoint
    """
    seed_everything(SEED)
    metrics_array = np.zeros(len(gen_dfs), dtype=np.float32)
    success_array = np.ones(len(gen_dfs), dtype=np.bool8)

    test_ds = real_ds.df_to_endpoint_tensor_dataset(
        real_ds.sample_to_df(
            real_ds.test_dataset[:][0].cpu(),
            real_ds.test_dataset[:][1].cpu(),
            real_ds.test_dataset[:][-1].cpu(),
            transform_categorical=False,
            transform_continuous=False,
        ),
        180,
        endpoint,
        scale_time_delta=True,
    )

    with tqdm(total=len(gen_dfs), desc=f"Endpoint {endpoint}", position=endpoint-1, leave=True) as pbar:
        for i, gen_df in enumerate(gen_dfs):
            try:
                (
                    synthetic_static_tensor,
                    synthetic_temporal_tensor,
                    synthetic_seq_lengths_tensor,
                ) = real_ds.patient_dfs_to_tensors(
                    real_ds.encode_features(
                        gen_df,
                        reassign_seq_lengths_weights=True,
                        requires_median_delta_calc=False,
                    )
                )
                synthetic_temporal_tensor[:, 0, -1] = 0.0

                train_ds = real_ds.df_to_endpoint_tensor_dataset(
                    real_ds.sample_to_df(
                        synthetic_static_tensor,
                        synthetic_temporal_tensor,
                        synthetic_seq_lengths_tensor,
                        transform_categorical=False,
                        transform_continuous=False,
                    ),
                    180,
                    endpoint,
                    scale_time_delta=True,
                )

                p_metric = PrognosisMetric.compute(
                    train_ds,
                    test_ds,
                    enable_progress_bar=False,
                    data_loader_num_workers=mp.cpu_count() // 5,
                    data_loader_context="fork",
                )

                metrics_array[i] = p_metric
                
            except Exception as e:
                success_array[i] = False
                print(f"\nError processing endpoint {endpoint}, dataframe {i}: {str(e)}")
                
            finally:
                pbar.update(1)
    
    return metrics_array, success_array, endpoint

In [ ]:
prognostic_metrics = np.zeros((5, len(gen_dfs)), dtype=np.float32)
successful_prognostic_runs = np.ones((5, len(gen_dfs)), dtype=np.bool8)

parallel = Parallel(n_jobs=5, verbose=1)
results = parallel(
    delayed(process_endpoint_synthetic_data)(endpoint, deepcopy(gen_dfs), deepcopy(real_ds))
    for endpoint in range(1, 6)
)

for metrics, success, endpoint in results:
    prognostic_metrics[endpoint - 1] = metrics
    successful_prognostic_runs[endpoint - 1] = success

In [ ]:
with open(f"prognostic_metrics_patientflow_{version}.pkl", "wb") as f:
    pickle.dump((prognostic_metrics, successful_prognostic_runs), f)

In [ ]:
prognostic_metrics = []


with open(f"prognostic_metrics_patientflow_{version}.pkl", "rb") as f:
    _prognostic_metrics, successful_prognostic_runs = pickle.load(f)
    for endpoint in range(1, 6):
        print(
            f"[C{endpoint}] Success Rate: {successful_prognostic_runs[endpoint - 1].sum()}"
        )
    prognostic_metrics = _prognostic_metrics

In [ ]:
for endpoint in range(1, 6):
    print(f"[C{endpoint}] Min: {prognostic_metrics[endpoint - 1].min()}")
    print(
        f"[C{endpoint}] 25th percentile: {np.percentile(prognostic_metrics[endpoint - 1], 25)}"
    )
    print(f"[C{endpoint}] Median: {np.median(prognostic_metrics[endpoint - 1])}")
    print(
        f"[C{endpoint}] 75th percentile: {np.percentile(prognostic_metrics[endpoint - 1], 75)}"
    )
    print(f"[C{endpoint}] Max: {prognostic_metrics[endpoint - 1].max()}")
    print(f"[C{endpoint}] Mean: {prognostic_metrics[endpoint - 1].mean()}")
    print(f"[C{endpoint}] Std: {prognostic_metrics[endpoint - 1].std()}")
    print()

In [ ]:
dfs = []

dfs.append(
    pd.DataFrame(
        {
            "Value": msas_scores,
            "Metric": ["eMSAS"] * len(msas_scores),
            "Endpoint": [None] * len(msas_scores),
            "Dataset": [None] * len(msas_scores),
        }
    )
)

for endpoint in range(1, 6):
    dfs.append(
        pd.DataFrame(
            {
                "Value": prognostic_metrics[endpoint - 1],
                "Metric": ["Prognostic"] * len(prognostic_metrics[endpoint - 1]),
                "Endpoint": [f"C{endpoint}"] * len(prognostic_metrics[endpoint - 1]),
                "Dataset": "Synthetic",
            }
        )
    )
    dfs.append(
        pd.DataFrame(
            {
                "Value": real_prognostic_metrics[endpoint],
                "Metric": ["Prognostic"] * len(real_prognostic_metrics[endpoint]),
                "Endpoint": [f"C{endpoint}"] * len(real_prognostic_metrics[endpoint]),
                "Dataset": "Real",
            }
        )
    )

In [ ]:
numerical_feats = ["Age_onset", "Height (m)", "Weight"]

categorical_feats = [
    "Gender",
    "Ethnicity",
    "NIV",
    "Tracheostomy",
    "PEG",
    "UMNvsLMN",
    "Onset",
    "Limb_O",
    "Limbs_Impairment",
    "Limbs_Side",
    "Weightloss_>10%",
    "ALS_familiar_history",
    "Ever_smoked",
    "Blood_hypertension",
    "Diabetes",
    "Dyslipidemia",
    "Thyroid",
    "Autoimmune",
    "SOD1 Mutation",
    "Stroke",
    "Cardiac_disease",
    "Primary_cancer",
    "C9orf72",
    "TARDBP mutation",
    "FUS mutation",
]

In [ ]:
# CAT Metric
# V = Number of binary / categorical variables
# CAT(X, X_synth) = 1 / V * sum_{i=1}^V \frac{|X_synth^V|}{|X^V|}
# where |X^V| is the support (or number of unique values) of the variable V in X

cats = np.array(
    [
        (1 / len(categorical_feats))
        * sum(
            [
                len(gen_df[feat].unique()) / len(real[feat].unique())
                for feat in categorical_feats
            ]
        )
        for gen_df in gen_dfs
    ]
)

print(f"Min: {cats.min()}")
print(f"25th percentile: {np.percentile(cats, 25)}")
print(f"Median: {np.median(cats)}")
print(f"75th percentile: {np.percentile(cats, 75)}")
print(f"Max: {cats.max()}")
print(f"Mean: {cats.mean()}")
print(f"Std: {cats.std()}")
print()


In [ ]:
dfs.append(
    pd.DataFrame(
        {
            "Value": cats,
            "Metric": ["Support Coverage"] * len(cats),
            "Endpoint": [None] * len(cats),
            "Dataset": [None] * len(cats),
        }
    )
)

In [ ]:
df = pd.concat(dfs, ignore_index=True)

In [ ]:
sns.set(font_scale=1.1)

In [ ]:
# Set the style for better visuals
sns.set_style("whitegrid")

# Create a new custom column for plotting that combines Metric and Task
df["Plot_Category"] = df.apply(
    lambda row: row["Metric"] if row["Metric"] != "Prognostic" else row["Endpoint"],
    axis=1,
)

# Create a figure with proper dimensions
plt.figure(figsize=(14, 8))

# Define a custom order for our x-axis categories
custom_order = ["eMSAS"] + [f"C{i}" for i in range(1, 6)] + ["Support Coverage"]

# Create a palette that handles the None case for Metrics A and C
palette_dict = {"Synthetic": "salmon", "Real": "teal", None: "lightblue"}

boxprops = {
    "facecolor": "salmon",
    "edgecolor": "teal",
    "hatch": "///",
    "linewidth": 1.0,
}

ax = sns.boxplot(
    y="Value",
    data=df[df["Metric"] == "eMSAS"],
    width=0.4,  # Make it slightly narrower than the paired B boxes total width
    positions=[0],
    showfliers=False,
    boxprops=boxprops,
    medianprops={"color": "black", "linewidth": 1.0},
    whiskerprops={"color": "black", "linewidth": 1.0},
    capprops={"color": "black", "linewidth": 1.0},
)

sns.boxplot(
    x="Plot_Category",  # Use the category for internal grouping
    y="Value",
    hue="Dataset",
    data=df[df["Metric"] == "Prognostic"],
    order=[f"C{i}" for i in range(1, 6)],  # Order for the B plot part
    palette=palette_dict,
    width=0.7,  # Adjust width as needed
    showfliers=False,
    ax=ax,
)

sns.boxplot(
    y="Value",
    data=df[df["Metric"] == "Support Coverage"],
    width=0.4,  # Make it slightly narrower than the paired B boxes total width
    showfliers=False,
    positions=[6],
    ax=ax,
    boxprops=boxprops,
    medianprops={"color": "black", "linewidth": 1.0},
    whiskerprops={"color": "black", "linewidth": 1.0},
    capprops={"color": "black", "linewidth": 1.0},
)

plt.xlim(-0.5, 7.5)


# Add vertical separators between metric groups
def add_separator(ax, x_position):
    ax.axvline(x=x_position, color="gray", linestyle="--", alpha=0.7, ymin=0, ymax=1)


# Add separators between metric groups
add_separator(ax, 0.43)  # Between A and B tasks
add_separator(ax, 5.57)  # Between B tasks and C

plt.annotate(
    "eMSAS",
    xy=(0, -0.075),
    xycoords=("data", "axes fraction"),
    ha="center",
    va="center",
)

# Create the Metric B label
# First, find position
b_start = custom_order.index("C1")
b_end = custom_order.index("C5")
b_middle = (b_start + b_end) / 2

# Add the Metric B text label
plt.annotate(
    "Prognostic",
    xy=(b_middle, -0.075),
    xycoords=("data", "axes fraction"),
    ha="center",
    va="center",
)

plt.annotate(
    "Support\nCoverage",
    xy=(len(custom_order) - 1, -0.075),
    xycoords=("data", "axes fraction"),
    ha="center",
    va="center",
)


# Add title and labels
# plt.title('Comparison of Metrics Across Tasks and Datasets', fontsize=16)
plt.xlabel("", fontsize=12, labelpad=30)
plt.ylabel("Value", fontsize=12)

# Fix the legend - make sure it only shows datasets D and F
handles, labels = ax.get_legend_handles_labels()
new_handles = []
new_labels = []

# Filter the legend to only include D and F
for handle, label in zip(handles, labels):
    if label in ["Synthetic", "Real"]:
        new_handles.append(handle)
        new_labels.append(label)

# Create a new legend
plt.legend(new_handles, new_labels, title="Dataset", loc="upper right")

In [ ]:
ax.get_figure().savefig(f"boxplot_metrics_patientflow_{version}.pdf")